## Objetivo

Realizar a ingestão dos dados brutos de Airbnb armazenados no Volume
do Databricks para tabelas Delta na camada Bronze.

## Fontes

- Listings Data
- Past Calendar Rates
- Future Calendar Rates
- Reviews Data

## Estratégia

Os dados são carregados a partir dos arquivos Parquet armazenados
no Volume `bronze/raw_files` e persistidos como tabelas Delta
no schema `bronze`.

Nesta camada, os dados são preservados o mais próximo possível
da fonte original. Transformações analíticas e regras de negócio
serão aplicadas posteriormente na camada Silver.

In [0]:
catalog = "airbnb_joao_pessoa"
bronze_schema = "bronze"
raw_path = f"/Volumes/{catalog}/{bronze_schema}/raw_files"

In [0]:
# definindo nome das tabelas
tables = {
    "listings": f"{catalog}.{bronze_schema}.listings",
    "past_rates": f"{catalog}.{bronze_schema}.past_calendar_rates",
    "future_rates": f"{catalog}.{bronze_schema}.future_calendar_rates",
    "reviews": f"{catalog}.{bronze_schema}.reviews"
}

In [0]:
display(dbutils.fs.ls(raw_path))

In [0]:
# Leitura
listings = spark.read.parquet(f"{raw_path}/Listings_Data.parquet")
past_rates = spark.read.parquet(f"{raw_path}/Past_Calendar_Rates.parquet")
future_rates = spark.read.parquet(f"{raw_path}/Future_Calendar_Rates.parquet")
reviews = spark.read.parquet(f"{raw_path}/Reviews_Data.parquet")

In [0]:
print(f"Listings: {listings.count()}")
print(f"Past Calendar Rates: {past_rates.count()}")
print(f"Future Calendar Rates: {future_rates.count()}")
print(f"Reviews: {reviews.count()}")

In [0]:
# Verificar se não estão vazios
datasets = {
    "listings": listings,
    "past_rates": past_rates,
    "future_rates": future_rates,
    "reviews": reviews
}

for name, df in datasets.items():
    count = df.limit(1).count()
    if count == 0:
        raise ValueError(f"O dataset {name} está vazio.")

In [0]:
# Verififcar e validar coluna chave
required_columns = {
    "listings": ["listing_id"],
    "past_rates": ["listing_id", "date"],
    "future_rates": ["listing_id", "date"],
    "reviews": ["listing_id", "date"]
}

for name, columns in required_columns.items():
    df = datasets[name]

    missing = [c for c in columns if c not in df.columns]

    if missing:
        raise ValueError(
            f"{name}: colunas obrigatórias ausentes: {missing}"
        )


### Escrita das tabelas bronze

In [0]:
(
    listings.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tables["listings"])
)

In [0]:
(
    past_rates.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tables["past_rates"])
)

In [0]:
(
    future_rates.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tables["future_rates"])
)

In [0]:
(
    reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tables["reviews"])
)

### Validação

In [0]:
bronze_listings = spark.table(tables["listings"])
bronze_past_rates = spark.table(tables["past_rates"])
bronze_future_rates = spark.table(tables["future_rates"])
bronze_reviews = spark.table(tables["reviews"])

In [0]:
print("Listings:", bronze_listings.count())
print("Past Rates:", bronze_past_rates.count())
print("Future Rates:", bronze_future_rates.count())
print("Reviews:", bronze_reviews.count())